In [0]:
#Verificação de quantidade de municípios, tipos de coluna e período
df_pib = spark.table("mvp_1.bronze.pib_municipio")

display(df_pib)

print(f"Linhas: {df_pib.count()}")
print(f"Colunas: {len(df_pib.columns)}")

df_pib.printSchema()

Cód.,Município,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
1100015,Alta Floresta D'Oeste (RO),111291,143222,173991,167127,168805,191364,248962,256986,262077,280510,329029,341325,377799,421300,478217,485374,498980,495775,570242,734467,919520,1046343
1100023,Ariquemes (RO),449593,539636,657193,749021,790697,905203,1064822,1133095,1364694,1651885,1703642,1799853,1921532,2037799,2184346,2287910,2464704,2579278,2817331,3211294,3809355,4383815
1100031,Cabixi (RO),31768,40985,43392,49130,46884,49166,60588,69776,69611,77217,99586,96365,113477,116565,133342,138110,140503,139976,167153,238414,289783,300832
1100049,Cacoal (RO),474443,622437,622415,758960,743194,814890,928699,985479,1186494,1259024,1372705,1433254,1660650,1794478,1947283,2082761,2175840,2261930,2518845,2792506,3195489,3848468
1100056,Cerejeiras (RO),79174,99983,121366,129107,124415,143270,167474,190902,222021,260142,357333,353270,392417,397736,408194,439245,470647,506494,600630,743062,903099,1006389
1100064,Colorado do Oeste (RO),87254,103363,114815,126534,126670,137899,157026,174168,193093,206425,222071,242767,273759,285372,306518,328377,330232,334922,366922,424846,514219,580899
1100072,Corumbiara (RO),45165,57284,67309,70936,68935,72291,95990,119226,114768,145412,175872,168681,186868,190906,236624,332804,320416,349289,268353,396740,473845,509566
1100080,Costa Marques (RO),37308,50996,53905,62986,60812,74215,90971,99854,107583,120018,134091,149739,168605,187170,206101,212879,230151,238796,261944,316672,349796,413943
1100098,Espigão D'Oeste (RO),119312,153425,180676,203257,195271,220452,265616,291491,311788,347581,376177,426546,462389,483962,543318,566361,606072,624787,666321,773372,921382,1023973
1100106,Guajará-Mirim (RO),174680,247248,302287,327643,325665,374116,491320,521104,598167,715386,516172,605131,646154,682458,740352,775195,837459,892778,984614,1054255,1057883,1145034


Linhas: 5571
Colunas: 24
root
 |-- Cód.: string (nullable = true)
 |-- Município: string (nullable = true)
 |-- 2002: string (nullable = true)
 |-- 2003: string (nullable = true)
 |-- 2004: string (nullable = true)
 |-- 2005: string (nullable = true)
 |-- 2006: string (nullable = true)
 |-- 2007: string (nullable = true)
 |-- 2008: string (nullable = true)
 |-- 2009: string (nullable = true)
 |-- 2010: string (nullable = true)
 |-- 2011: string (nullable = true)
 |-- 2012: string (nullable = true)
 |-- 2013: long (nullable = true)
 |-- 2014: long (nullable = true)
 |-- 2015: long (nullable = true)
 |-- 2016: long (nullable = true)
 |-- 2017: long (nullable = true)
 |-- 2018: long (nullable = true)
 |-- 2019: long (nullable = true)
 |-- 2020: long (nullable = true)
 |-- 2021: long (nullable = true)
 |-- 2022: long (nullable = true)
 |-- 2023: long (nullable = true)



Validação da camada Bronze - PIB municipal

Verificação da qualidade e tipo de dados dos códigos municipais e se há valores ausentes.

In [0]:
#Renomear células para leitura simplificada
from pyspark.sql import functions as F

df_pib = (
    spark.table("mvp_1.bronze.pib_municipio")
    .withColumnRenamed("Cód.", "codigo_municipio_ibge")
    .withColumnRenamed("Município", "municipio")
)

# Verificação de códigos municipais duplicados
duplicados = (
    df_pib
    .groupBy("codigo_municipio_ibge")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Verificação de códigos ou municípios ausentes
identificacao_ausente = (
    df_pib
    .filter(
        F.col("codigo_municipio_ibge").isNull() |
        F.col("municipio").isNull()
    )
    .count()
)

# Identificação das colunas referentes aos anos
colunas_anos = [c for c in df_pib.columns if c.isdigit()]

# Verificação de valores ausentes nos anos
for ano in colunas_anos:
    ausentes = (
        df_pib
        .filter(
            F.col(ano).isNull() |
            (F.trim(F.col(ano).cast("string")) == "") |
            (F.col(ano).cast("string") == "-")
        )
        .count()
    )

    if ausentes > 0:
        print(f"{ano}: {ausentes} valores ausentes")

print(f"Códigos duplicados: {duplicados}")
print(f"Identificação ausente: {identificacao_ausente}")
print(f"Período disponível: {min(colunas_anos)}-{max(colunas_anos)}")

2002: 1 valores ausentes
2003: 1 valores ausentes
2004: 1 valores ausentes
2005: 1 valores ausentes
2006: 1 valores ausentes
2007: 1 valores ausentes
2008: 1 valores ausentes
2009: 1 valores ausentes
2010: 1 valores ausentes
2011: 1 valores ausentes
2012: 1 valores ausentes
2013: 1 valores ausentes
2014: 1 valores ausentes
2015: 1 valores ausentes
2016: 1 valores ausentes
2017: 1 valores ausentes
2018: 1 valores ausentes
2019: 1 valores ausentes
2020: 1 valores ausentes
2021: 1 valores ausentes
2022: 1 valores ausentes
2023: 1 valores ausentes
Códigos duplicados: 0
Identificação ausente: 1
Período disponível: 2002-2023


In [0]:
# Identificar o registro ausente

display(
    df_pib.filter(
        F.col("codigo_municipio_ibge").isNull() |
        F.col("municipio").isNull()
    )
)

codigo_municipio_ibge,municipio,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
"Fonte: IBGE, em parceria com os Órgãos Estaduais de Estatística, Secretarias Estaduais de Governo e Superintendência da Zona Franca de Manaus - SUFRAMA",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


Foi identificado um registro referente à fonte de dados, não tendo a ver com a granularidade. Ele será desconsiderado na camada Silver.
